# 1.1 章节介绍

本节是 PyPTO 初级教程中“计算算子样例”章节的导学内容。后续章节会深入学习逐元素算子及其 API 全量覆盖、矩阵乘法、规约算子及其 API 全量覆盖、Tiling 及其 API 全量覆盖、形状变换和 Softmax。本节先介绍 Tensor 描述对象，然后用一个最小 kernel 建立共同的阅读方法。

一个 PyPTO 示例通常同时包含两种代码：kernel 外部的 PyTorch 代码，以及 kernel 内部的 PyPTO 计算描述。前者负责创建真实数据、选择设备、分配输出和验证结果；后者负责描述要编译执行的 Tensor 计算。初学时只要把这两层分清，很多代码就不再混乱。

本节围绕一个最小算子 `out = (a + b) * 2` 展开。学习时重点观察四件事：

1. host 侧如何创建真实的 `torch.Tensor`。
2. kernel 函数参数里的 `pypto.Tensor` 如何描述参数。
3. kernel 为什么把结果写入调用者传入的 `out`。
4. 验证代码如何用 PyTorch 参考结果证明 PyPTO kernel 算对了。

后续章节中的算子会越来越复杂，但阅读方法保持一致：先看输入输出，再看 shape 和 dtype，再看计算表达式，最后看验证方式。

---
## 1. 环境准备

正式开始实践之前，先初始化 Notebook 运行环境。本教程中的代码默认在已经完成 PyPTO、PyTorch NPU 和 CANN 环境配置的机器上运行。

本节代码优先在 NPU 上运行；当前环境没有 NPU 时，验证部分会自动跳过，仍然可以作为教材阅读。


In [ ]:
import os

# ---- 设备选择说明 ----
# PyPTO 教程使用两个环境变量控制 NPU 设备选择：
#
# 1. ASCEND_RT_VISIBLE_DEVICES：指定当前进程可见的物理 NPU 卡号。
#    必须在 import torch_npu 之前设置才生效。
#    设置后，可见的物理卡会被重新编号，逻辑编号从 0 开始。
#    例如 "13" 表示只让物理卡 13 可见，它在进程内变成逻辑 npu:0。
#    "5,8" 表示让物理卡 5 和 8 可见，它们分别变成逻辑 npu:0 和 npu:1。
#    保持 None 表示不限制，所有物理卡都可见。
#
# 2. TILE_FWK_DEVICE_ID：在可见卡中选择使用哪一张逻辑设备。
#    默认值为 "0"，即使用第一张可见卡。
#    如果可见卡不止一张，可以设置为 "1"、"2" 等选其他逻辑设备。
#
# 两者的关系：ASCEND_RT_VISIBLE_DEVICES 决定「哪些卡可见 + 映射规则」，
# TILE_FWK_DEVICE_ID 决定「在可见卡中选哪一张」。
# ASCEND_RT_VISIBLE_DEVICES = "13"
# ASCEND_RT_VISIBLE_DEVICES = "13"
# if ASCEND_RT_VISIBLE_DEVICES is not None:
#     os.environ["ASCEND_RT_VISIBLE_DEVICES"] = str(ASCEND_RT_VISIBLE_DEVICES)

import torch
import pypto

try:
    import torch_npu
except ImportError:
    torch_npu = None

def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass

def get_device():
    if torch_npu is None:
        print("torch_npu is not available; NPU execution will be skipped.")
        return "cpu"
    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    torch.npu.set_device(device_id)
    return f"npu:{device_id}"

reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

# print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)
print("pypto:", pypto.__file__)


这段单元只做运行环境准备。前几行导入 PyTorch、PyPTO 和可选的 `torch_npu`；`get_device()` 负责判断当前是否能使用 NPU，并把设备统一保存到 `device`。

### 两个设备环境变量的关系

PyPTO 教程使用两个环境变量配合控制 NPU 设备选择：

| 环境变量 | 作用 | 设置时机 | 示例 |
| --- | --- | --- | --- |
| `ASCEND_RT_VISIBLE_DEVICES` | 指定哪些**物理卡**可见，并在进程内重新编号 | 必须在 `import torch_npu` 之前 | `"13"` → 只有物理卡13可见，逻辑编号为 `npu:0` |
| `TILE_FWK_DEVICE_ID` | 在可见卡中选择**哪一张逻辑设备** | 任意时机，默认 `"0"` | `"0"` → 使用第一张可见卡 |

两者的关系可以这样理解：

- `ASCEND_RT_VISIBLE_DEVICES` 先决定「哪些物理卡可见 + 如何重映射」；可见的物理卡会被重新编号，逻辑编号从 0 开始。
- `TILE_FWK_DEVICE_ID` 再在映射后的逻辑编号中选择一张。默认是 `"0"`，即第一张可见卡。

所以代码中的 `device = "npu:0"` 代表的是「第一张可见卡」，不一定就是物理卡0。具体是哪张物理卡，取决于 `ASCEND_RT_VISIBLE_DEVICES` 的设置。

**注意事项**：修改 `ASCEND_RT_VISIBLE_DEVICES` 后需要重启 Jupyter kernel，因为 `torch_npu` 在初始化时就读取了这个变量，之后再改不会生效。

运行后需要关注输出中的几项：

```text
ASCEND_RT_VISIBLE_DEVICES: ...
TILE_FWK_DEVICE_ID: ...
device: npu:0
pypto: ...
```

如果 `device` 显示为 `npu:0`，说明后续验证单元会尝试在 NPU 上执行 kernel；如果显示为 `cpu`，说明当前环境没有可用的 NPU，验证会跳过。`pypto` 路径用于确认当前导入的是哪个 PyPTO 包。

这里的环境准备属于 host 侧逻辑，还没有进入 PyPTO kernel。后面的 JIT 函数才是要被 PyPTO 编译的计算描述。


## 2. 本章学习目标

本章围绕 PyPTO 初级阶段最常用的计算算子展开，目标不是一次性讲完所有 API，而是建立可迁移的开发方法。

完成本章后，应能够：

1. 看懂一个 PyPTO 示例中 host 侧代码和 kernel 侧代码的分工。
2. 使用 `pypto.set_vec_tile_shapes` 编写逐元素和规约类算子。
3. 使用 `pypto.set_cube_tile_shapes` 编写矩阵乘法类算子。
4. 理解 `dim`、`keepdim` 在规约计算中的作用。
5. 理解 Tiling 分块为什么存在，以及它和数学结果的关系。
6. 使用 view、gather、scatter、concat、transpose、cast 等操作组织 Tensor。
7. 将多个基础操作组合成一个行 Softmax 算子，并完成验证。
8. 理解 `pypto.Tensor` 和 `pypto.tensor` 的描述对象属性（shape、dtype、format、name）。
9. 使用 `pypto.arange`、`pypto.full` 等 API 在 kernel 内创建 Tensor。
10. 按模式阅读 `abs/add/sub/mul/div/clip/exp/log/sqrt` 等逐元素 API 示例。
11. 系统验证 `sum/amax/amin` 在不同 `dim` 和 `keepdim` 设置下的 shape 变化。


## 3. Tensor 描述与创建

在进入 JIT kernel 之前，先单独看一个 `pypto.Tensor` 描述对象。这个例子对应 quick-start 中的 Tensor Creation 部分，用来观察 Tensor 的 `name`、`shape`、`dtype`、`format` 和 `dim` 等属性。

`pypto.Tensor`（大写 T）是 PyPTO 的 Tensor 描述构造器，用来创建一个描述对象——指定 shape、dtype、name 等规格信息，但不持有真实数据。后续定义 kernel 函数参数时会频繁使用它。

PyPTO 还提供了 `pypto.tensor`（小写 t）工厂函数，功能与 `pypto.Tensor` 类似，但额外支持 `format` 参数。下面 §3.1 和 §3.2 的示例就使用了 `pypto.tensor`。两者创建的都是描述对象，不是 `torch.Tensor` 这种持有真实数据的对象。

In [ ]:
import pypto

tensor = pypto.Tensor([4, 4], pypto.DT_FP16, "my_tensor")
print("name:", tensor.name)
print("shape:", tensor.shape)
print("dtype:", tensor.dtype)
print("format:", tensor.format)
print("dim:", tensor.dim)


这个小例子只做属性展示，不涉及 NPU 执行，也不需要输出 Tensor。它的价值是帮助区分两类对象：

- `pypto.Tensor` / `pypto.tensor`（大写或小写）都是 **描述对象**，负责声明 kernel 看到的 Tensor 规格（shape、dtype、format、name），不持有真实数据。
- `torch.Tensor` 是 **真实数据对象**，host 侧创建并传入 kernel，包含实际数值。

初学时记住：kernel 函数参数里写的是描述，调用时传的是数据。

### 3.1 使用不同 dtype 创建 Tensor

下面遍历常见 PyPTO dtype，创建 shape 为 `[2, 3]` 的 Tensor 描述对象，并打印名称和 dtype。

In [ ]:
import pypto

data_types = [
    (pypto.DT_INT4, "DT_INT4"),
    (pypto.DT_INT8, "DT_INT8"),
    (pypto.DT_INT16, "DT_INT16"),
    (pypto.DT_INT32, "DT_INT32"),
    (pypto.DT_INT64, "DT_INT64"),
    (pypto.DT_FP8, "DT_FP8"),
    (pypto.DT_FP16, "DT_FP16"),
    (pypto.DT_FP32, "DT_FP32"),
    (pypto.DT_BF16, "DT_BF16"),
    (pypto.DT_HF4, "DT_HF4"),
    (pypto.DT_HF8, "DT_HF8"),
    (pypto.DT_UINT8, "DT_UINT8"),
    (pypto.DT_UINT16, "DT_UINT16"),
    (pypto.DT_UINT32, "DT_UINT32"),
    (pypto.DT_UINT64, "DT_UINT64"),
    (pypto.DT_BOOL, "DT_BOOL"),
]

for dtype, dtype_name in data_types:
    tensor = pypto.tensor([2, 3], dtype, f"tensor_{dtype_name}")
    print(f"{dtype_name}: name={tensor.name}, dtype={tensor.dtype}")


### 3.2 指定 Tensor format

上一节用 `pypto.Tensor`（大写 T）创建描述对象。本节改用 `pypto.tensor`（小写 t），它除了 shape、dtype、name 之外，还支持第四个参数 `format`。下面使用 `pypto.TileOpFormat.TILEOP_NZ` 创建 Tensor 描述。

In [ ]:
import pypto

tensor = pypto.tensor([512, 32], pypto.DT_FP16, "sparse_tensor", pypto.TileOpFormat.TILEOP_NZ)

print("Shape:", tensor.shape)
print("Data Type:", tensor.dtype)
print("Dimensions:", tensor.dim)
print("Format:", tensor.format)
print("Name:", tensor.name)


## 4. PyPTO 算子的基本组成

一个 PyPTO 算子通常包含 host 侧和 kernel 侧两部分。host 侧负责准备真实数据，kernel 侧负责描述要编译执行的 Tensor 计算。

| 组成部分 | 出现位置 | 作用 |
| --- | --- | --- |
| `torch.Tensor` 输入输出 | kernel 外部 | 真实数据，位于 CPU 或 NPU 设备上 |
| JIT 装饰器 | kernel 定义处 | 标记这个 Python 函数需要进入 PyPTO 编译流程 |
| `pypto.Tensor(...)` 参数 | kernel 函数参数 | 描述输入输出 Tensor 的 dtype、shape 或动态特征 |
| Tile Shape 设置 | kernel 内部 | 告诉编译系统如何组织向量或矩阵类计算 |
| 输出写回 | kernel 内部 | 将计算表达式的最终结果写入调用者传入的输出 Tensor |

下面给出一个最小示例。它的数学含义可以概括为：

```python
out = (a + b) * 2
```

但从 PyPTO 的角度看，这个函数不只是普通 Python 函数，而是一段可以被编译的 Tensor 计算描述。
在 Notebook 中，完整示例会先清理 PyPTO 的记录状态，再定义当前模块的 JIT kernel。这样前一次失败运行不会影响本单元的验证。


In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_mul_kernel(
    a: pypto.Tensor([], pypto.DT_FP16),
    b: pypto.Tensor([], pypto.DT_FP16),
    out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.mul(pypto.add(a, b), 2.0))

def main_add_mul():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return
    a = torch.randn((8, 8), dtype=torch.float16, device=device)
    b = torch.randn((8, 8), dtype=torch.float16, device=device)
    out = torch.empty_like(a)

    add_mul_kernel(a, b, out)
    ref = (a + b) * 2.0

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-2, atol=1e-2)
    print("add_mul_kernel 验证通过")
    print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
    print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
    print("device:", device, "run_mode:", RUN_MODE)
    print("输入 shape:", tuple(a.shape), "输出 shape:", tuple(out.shape))
    print("最大误差:", max_diff)
    print("输出前 2x4:", out[:2, :4].cpu())

main_add_mul()


`add_mul_kernel` 把两个逐元素步骤串在一起：先计算 `a + b`，再乘以 `2.0`。每一步的输入输出 shape 都保持一致，最后通过 `out.move(...)` 写回结果。

这里有一个初学时最容易混淆的地方：函数参数里的 `a`、`b`、`out` 写成 `pypto.Tensor`，表示它们是 kernel 参数描述；真正的数据在 `main_add_mul()` 中用 `torch.randn` 创建，并在调用 `add_mul_kernel(a, b, out)` 时传进来。

因此，`pypto.Tensor` 更像“这个 kernel 接受什么样的 Tensor”的约定，`torch.Tensor` 才是运行时实际参与计算的数据。这个代码单元也包含 host 侧验证逻辑，能够独立完成输入创建、kernel 调用、参考结果计算和误差检查。


## 5. 代码逐行理解

上面的代码虽然很短，但已经包含 PyPTO 算子开发的基本骨架。

- `@pypto.frontend.jit(...)` 表示该函数会进入 PyPTO 的编译流程。它看起来像 Python 函数，但执行方式已经不再是普通解释执行。
- `pypto.Tensor([], pypto.DT_FP16)` 表示输入或输出是 FP16 Tensor。这里的空 shape 让示例更关注计算表达本身，运行时会根据传入的 `torch.Tensor` 确定实际 shape。
- `pypto.set_vec_tile_shapes(8, 8)` 表示这段向量类计算按二维 Tile 组织。它不改变 `out = (a + b) * 2` 的数学结果，只影响底层执行如何分块。
- `pypto.add(a, b)` 描述逐元素加法，`pypto.mul(..., 2.0)` 描述逐元素乘标量。
- `out.move(...)` 表示将计算结果写入输出 Tensor。输出 Tensor 由 host 侧提前分配，并作为参数传入 kernel。

可以把这个 kernel 翻译成自然语言：

```text
接收两个 FP16 Tensor a 和 b。
按照 vec tile 组织逐元素计算。
计算 (a + b) * 2.0。
把结果写入调用者传入的 out。
```

需要注意，Tile Shape 不是数学公式的一部分。它解决的是“计算如何在 NPU 上组织执行”的问题，而不是“结果应该是什么”的问题。

## 6. 运行与验证

完整代码单元中，host 侧会创建真实的 `torch.Tensor`，然后像调用函数一样调用 kernel：

```python
add_mul_kernel(a, b, out)
```

这里的 `a`、`b`、`out` 都是真实 Tensor。调用发生后，kernel 把计算结果写进已经分配好的 `out`，所以示例中常见的模式是：

```python
out = torch.empty_like(a)
add_mul_kernel(a, b, out)
```

而不是：

```python
out = add_mul_kernel(a, b)
```

这种写法让输出 shape、dtype 和 device 在 host 侧明确下来，kernel 专心描述“怎么计算并写回”。

任何自定义算子都应该有验证闭环。初学阶段始终使用 PyTorch 参考实现检查结果：

1. 构造输入 Tensor。
2. 提前分配输出 Tensor `out`。
3. 调用 PyPTO kernel 写入 `out`。
4. 使用 PyTorch 写出参考结果 `ref`。
5. 比较 `out` 和 `ref` 的最大误差，并用 `torch.testing.assert_close` 做正式检查。

最大误差常写成：

```python
max_diff = (out - ref).abs().max().item()
```

它表示所有位置中最大的绝对误差：先逐元素相减，再取绝对值，再取最大值，最后用 `.item()` 转成 Python 数字。这个值越接近 0，说明 PyPTO kernel 和 PyTorch 参考实现越一致。


上面的完整单元用来验证 `add_mul_kernel`。它先在当前设备上构造两个小尺寸输入，再创建 `out = torch.empty_like(a)` 作为输出缓冲区。调用 `add_mul_kernel(a, b, out)` 后，`out` 中保存 PyPTO kernel 的计算结果。

随后，代码用普通 PyTorch 写出等价参考结果：

```python
ref = (a + b) * 2.0
```

最后打印 device、run_mode、shape、样例值和最大误差，并用 `assert_close` 做正式检查。误差超出阈值时会直接报错，这比只看打印结果更可靠。


**预期输出说明**

运行成功后，会打印验证通过信息，并显示输入/输出 shape、最大误差以及输出 Tensor 的前几项。具体数值每次随机生成，可能不同，但判断重点不在具体数值，而在以下几项：

```text
add_mul_kernel 验证通过
输入 shape: ...
输出 shape: ...
最大误差: ...
```

输入和输出 shape 应该一致，最大误差应该在阈值范围内。看到这些信息，就可以确认本节的最小 PyPTO kernel 已完成从输入、计算、写回到验证的闭环。

## 7. 本节自测

1. PyPTO JIT 函数和普通 Python 函数的核心区别是什么？
2. kernel 函数参数里的 `pypto.Tensor` 和调用时传入的 `torch.Tensor` 有什么区别？
3. 为什么示例要先创建 `out`，再调用 `add_mul_kernel(a, b, out)`？
4. `set_vec_tile_shapes` 会改变数学结果吗？
5. `(out - ref).abs().max().item()` 表示什么？

参考答案：

1. PyPTO JIT 函数会进入 PyPTO 编译流程，普通 Python 函数只是解释执行。
2. `pypto.Tensor` 是 kernel 参数描述，`torch.Tensor` 是真实输入输出数据。
3. PyPTO kernel 通常把结果写入调用者提前分配的输出 Tensor。
4. 不会。Tile Shape 影响执行组织方式，不改变数学语义。
5. 表示 PyPTO 输出和 PyTorch 参考结果之间的最大绝对误差。


## 8. 本节小结

本节完成了 PyPTO 算子开发的入门铺垫。一个最小算子包含 host 侧真实数据、kernel 侧参数描述、JIT 编译入口、Tile Shape 设置、输出写回和 PyTorch 参考验证。

需要记住：`pypto.Tensor` / `pypto.tensor` 描述 Tensor 规格（shape、dtype、format、name），与 host 侧的 `torch.Tensor` 是不同层次的对象。kernel 作者关注的是输入输出 Tensor 的规格、计算表达式和验证逻辑。

后续章节会沿着这个基本结构继续展开：逐元素算子使用向量 Tile，矩阵乘法使用 Cube Tile，规约算子需要理解维度含义，最后将这些基础能力组合成 Softmax。
